#CVIU Experiment 2:D1 Leave-One-Illumination-Out

This notebook evaluates six methods under four leave-one-illumination-out conditions.

For each condition:

- one illumination is used as the test subset
- the remaining three illuminations are used as the training subset
- Naive Prediction is recalculated from the corresponding training-subset mean distribution

The evaluated methods are:

- **Naive Prediction**
- **ED Regression**
- **M1:Siamese**
- **M2:Siamese+Lab**
- **M3:Siamese+ΔE**
- **M4:Siamese+ΔE+Lab**

All four illumination conditions and all six methods are executed in one code cell. The final output is a single table with methods as rows and held-out illumination conditions as columns.


##1.Setup

In [15]:
import gc
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from scipy.optimize import curve_fit
from IPython.display import display, Markdown

warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
)

print("PyTorch:", torch.__version__)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

PyTorch: 2.11.0+cu128
Device: cuda


##2.Parameters

This cell contains the configurable file paths, illumination identification, preprocessing, architecture, training, direct-Lab weighting, scheduler, regression, and output parameters.

In [ ]:
#File paths
D1_PATH = (
    ""
) # Please Fill the File Path

OUTPUT_DIR = Path(
    ""
) # Please Fill the File Path
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

#Random seed
SEED = 42

#Illumination identification
#
# Set the exact column name when D1 contains an illumination column.
# When None, common names are detected automatically.
# If no column is detected and D1 has 800 rows, the fallback assumes
# four consecutive blocks of 200 rows.
ILLUMINATION_COLUMN = None

FALLBACK_ILLUMINATION_LABELS = [
    "Light 1",
    "Light 2",
    "Light 3",
    "Light 4",
]

#Preprocessing
#Options: "full_d1" or "training_only"
NORMALIZATION_SCOPE = "full_d1"

#Architecture
ENCODER_LAYERS = [
    32,
    64,
    64,
    64,
    64,
]

REGRESSOR_LAYERS = [
    32,
    32,
    16,
    8,
]

#Training
EPOCHS = 300
BATCH_SIZE = 8
LEARNING_RATE = 5e-4
INPUT_NOISE_STD = 0.01
WEIGHT_DECAY = 0.0
PRINT_EVERY = 25

#Direct Lab-feature weights
M2_LAB_WEIGHT = 0.01
M4_LAB_WEIGHT = 0.01

#Learning-rate scheduler
USE_SCHEDULER = True
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 50
SCHEDULER_THRESHOLD = 1e-3
MIN_LEARNING_RATE = 1e-6

#Regression and numerical stability
REGRESSION_MAX_FUNCTION_EVALUATIONS = 20000
EPSILON = 1e-7

#Output
SAVE_MODEL_WEIGHTS = True
SAVE_PREDICTIONS = True

METHOD_ORDER = [
    "Naive Prediction",
    "ED Regression",
    "Siamese",
    "Siamese+Lab",
    "Siamese+ΔE",
    "Siamese+ΔE+Lab",
]

print("Output directory:", OUTPUT_DIR)

Output directory: /content/drive/MyDrive/CVIU_Experiment2_Leave_One_Illumination_Out


##3.Data preparation

In [17]:
def read_csv_auto(path):
    return pd.read_csv(
        path,
        sep=None,
        engine="python",
        header=0,
    )


def normalize_response_values(response_array):
    values = {
        value
        for value in pd.unique(
            response_array.ravel()
        )
        if not pd.isna(value)
    }

    if values.issubset({0, 1, 2}):
        return response_array

    if values.issubset({1, 2, 3}):
        return response_array - 1

    raise ValueError(
        "Observer responses must use either "
        "{0,1,2} or {1,2,3}. "
        f"Detected values: {sorted(values)}"
    )


def add_probability_columns(
    dataframe,
    response_columns,
):
    dataframe = dataframe.copy()

    probability_sets = [
        (
            "possibility0",
            "possibility1",
            "possibility2",
        ),
        (
            "probability0",
            "probability1",
            "probability2",
        ),
        (
            "p0",
            "p1",
            "p2",
        ),
        (
            "true1",
            "true2",
            "true3",
        ),
    ]

    for columns in probability_sets:
        if all(
            column in dataframe.columns
            for column in columns
        ):
            probabilities = dataframe.loc[
                :,
                columns,
            ].to_numpy(dtype=np.float32)

            probabilities = probabilities / np.clip(
                probabilities.sum(
                    axis=1,
                    keepdims=True,
                ),
                EPSILON,
                None,
            )

            dataframe["possibility0"] = probabilities[:, 0]
            dataframe["possibility1"] = probabilities[:, 1]
            dataframe["possibility2"] = probabilities[:, 2]
            return dataframe

    if not response_columns:
        raise ValueError(
            "No observer-response columns "
            "or probability columns were found."
        )

    responses = dataframe.loc[
        :,
        response_columns,
    ].to_numpy(dtype=np.float32)

    responses = normalize_response_values(
        responses
    )

    denominator = len(response_columns)

    for category in [0, 1, 2]:
        dataframe[
            f"possibility{category}"
        ] = (
            responses == category
        ).sum(axis=1) / denominator

    return dataframe


def detect_feature_columns(
    dataframe,
    response_columns,
):
    explicit_sets = [
        [
            "L1",
            "a1",
            "b1",
            "L2",
            "a2",
            "b2",
        ],
        [
            "L_1",
            "a_1",
            "b_1",
            "L_2",
            "a_2",
            "b_2",
        ],
        [
            "L1*",
            "a1*",
            "b1*",
            "L2*",
            "a2*",
            "b2*",
        ],
    ]

    for columns in explicit_sets:
        if all(
            column in dataframe.columns
            for column in columns
        ):
            return columns

    excluded_columns = {
        "C1",
        "C2",
        "Average",
        "possibility0",
        "possibility1",
        "possibility2",
        *response_columns,
    }

    numeric_candidates = [
        column
        for column in dataframe.columns
        if column not in excluded_columns
        and pd.api.types.is_numeric_dtype(
            dataframe[column]
        )
    ]

    if len(numeric_candidates) < 6:
        raise ValueError(
            "Could not identify six Lab columns. "
            f"Candidates: {numeric_candidates}"
        )

    return numeric_candidates[:6]


def prepare_xy(
    dataframe,
    response_columns,
):
    prepared = add_probability_columns(
        dataframe,
        response_columns,
    )

    feature_columns = detect_feature_columns(
        prepared,
        response_columns,
    )

    X = prepared.loc[
        :,
        feature_columns,
    ].to_numpy(dtype=np.float32)

    Y = prepared.loc[
        :,
        [
            "possibility0",
            "possibility1",
            "possibility2",
        ],
    ].to_numpy(dtype=np.float32)

    Y = Y / np.clip(
        Y.sum(
            axis=1,
            keepdims=True,
        ),
        EPSILON,
        None,
    )

    return (
        X,
        Y,
        prepared,
        feature_columns,
    )


def infer_illumination_labels(dataframe):
    if ILLUMINATION_COLUMN is not None:
        if ILLUMINATION_COLUMN not in dataframe.columns:
            raise ValueError(
                f"{ILLUMINATION_COLUMN!r} was not found in D1."
            )

        labels = dataframe[
            ILLUMINATION_COLUMN
        ].astype(str).to_numpy()

        if len(pd.unique(labels)) != 4:
            raise ValueError(
                "The selected illumination column "
                "must contain exactly four conditions."
            )

        return labels, ILLUMINATION_COLUMN

    common_names = [
        "illumination",
        "Illumination",
        "ILLUMINATION",
        "light",
        "Light",
        "LIGHT",
        "lighting",
        "Lighting",
        "condition",
        "Condition",
    ]

    for column in common_names:
        if (
            column in dataframe.columns
            and dataframe[column].nunique(
                dropna=False
            ) == 4
        ):
            return (
                dataframe[column].astype(str).to_numpy(),
                column,
            )

    if len(dataframe) == 800:
        labels = np.concatenate([
            np.repeat(
                FALLBACK_ILLUMINATION_LABELS[index],
                200,
            )
            for index in range(4)
        ])

        return (
            labels,
            "fallback: four consecutive blocks of 200 rows",
        )

    raise ValueError(
        "No illumination column was detected. "
        "Set ILLUMINATION_COLUMN manually."
    )


d1_dataframe = read_csv_auto(
    D1_PATH
)

d1_response_columns = [
    str(index)
    for index in range(1, 21)
    if str(index) in d1_dataframe.columns
]

(
    X_raw,
    Y,
    d1_prepared,
    FEATURE_COLUMNS,
) = prepare_xy(
    d1_dataframe,
    d1_response_columns,
)

assert len(X_raw) == 800, (
    f"Expected 800 D1 samples, got {len(X_raw)}."
)

(
    ILLUMINATION_LABELS,
    ILLUMINATION_SOURCE,
) = infer_illumination_labels(
    d1_prepared
)

UNIQUE_ILLUMINATIONS = list(
    pd.unique(
        ILLUMINATION_LABELS
    )
)

if len(UNIQUE_ILLUMINATIONS) != 4:
    raise ValueError(
        "Exactly four illumination conditions are required."
    )

illumination_overview = (
    pd.DataFrame({
        "Illumination": ILLUMINATION_LABELS
    })
    .value_counts()
    .rename("Sample count")
    .reset_index()
)

print("Feature columns:", FEATURE_COLUMNS)
print("Illumination source:", ILLUMINATION_SOURCE)
display(illumination_overview)

if "fallback" in ILLUMINATION_SOURCE:
    print(
        "WARNING: confirm that D1 is ordered as four "
        "consecutive illumination blocks of 200 rows."
    )

Feature columns: ['L1', 'a1', 'b1', 'L2', 'a2', 'b2']
Illumination source: fallback: four consecutive blocks of 200 rows


,Illumination,Sample count
0,Light 1,200
1,Light 2,200
2,Light 3,200
3,Light 4,200


##4.Shared components

In [18]:
class ColorPairDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        noise_std=0.0,
    ):
        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )
        self.Y = torch.tensor(
            Y,
            dtype=torch.float32,
        )
        self.noise_std = float(
            noise_std
        )

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        x = self.X[index].clone()

        if self.noise_std > 0:
            x = (
                x
                + torch.randn_like(x)
                * self.noise_std
            )

        return x, self.Y[index]


class SoftCrossEntropyLoss(nn.Module):

    def forward(
        self,
        logits,
        targets,
    ):
        log_probabilities = torch.log_softmax(
            logits,
            dim=1,
        )

        return -(
            targets
            * log_probabilities
        ).sum(dim=1).mean()


criterion = SoftCrossEntropyLoss()


class SharedEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        modules = []
        input_size = 3

        for index, output_size in enumerate(
            ENCODER_LAYERS
        ):
            modules.append(
                nn.Linear(
                    input_size,
                    output_size,
                )
            )

            if index < len(ENCODER_LAYERS) - 1:
                modules.append(
                    nn.ReLU()
                )

            input_size = output_size

        self.network = nn.Sequential(
            *modules
        )

    def forward(self, x):
        return self.network(x)


def build_regressor(input_size):
    modules = []
    current_size = input_size

    for hidden_size in REGRESSOR_LAYERS:
        modules.append(
            nn.Linear(
                current_size,
                hidden_size,
            )
        )
        modules.append(
            nn.ReLU()
        )
        current_size = hidden_size

    modules.append(
        nn.Linear(
            current_size,
            3,
        )
    )

    return nn.Sequential(
        *modules
    )


def calculate_siamese_distance(
    encoder,
    x,
):
    color1 = x[:, :3]
    color2 = x[:, 3:]

    embedding1 = encoder(color1)
    embedding2 = encoder(color2)

    return torch.linalg.norm(
        embedding1 - embedding2,
        dim=1,
        keepdim=True,
    )


def make_loader(
    X,
    Y,
    shuffle,
    noise_std,
    seed,
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        ColorPairDataset(
            X,
            Y,
            noise_std=noise_std,
        ),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )


def cross_entropy_numpy(
    targets,
    predictions,
):
    predictions = np.clip(
        predictions,
        EPSILON,
        1.0,
    )

    return float(
        np.mean(
            -np.sum(
                targets
                * np.log(predictions),
                axis=1,
            )
        )
    )

##5.Method definitions

###5-1.Naive Prediction

In [19]:
def run_naive(
    Y_training,
    Y_testing,
):
    mean_distribution = Y_training.mean(
        axis=0
    )

    predictions = np.repeat(
        mean_distribution.reshape(1, -1),
        len(Y_testing),
        axis=0,
    )

    return (
        cross_entropy_numpy(
            Y_testing,
            predictions,
        ),
        predictions.astype(np.float32),
        mean_distribution.astype(np.float32),
    )

###5-2.ED Regression

In [20]:
def nonlinear_regression_function(
    x,
    coefficient,
    exponent_coefficient,
):
    safe_x = np.maximum(
        x,
        1e-16,
    )

    exponent = np.clip(
        exponent_coefficient / safe_x,
        -8.0,
        8.0,
    )

    return coefficient * np.exp(exponent)


def make_probability_distribution(
    raw_predictions,
):
    predictions = np.asarray(
        raw_predictions,
        dtype=np.float64,
    )

    predictions = np.clip(
        predictions,
        EPSILON,
        None,
    )

    predictions = predictions / np.clip(
        predictions.sum(
            axis=1,
            keepdims=True,
        ),
        EPSILON,
        None,
    )

    return predictions.astype(
        np.float32
    )


def run_regression(
    X_training_raw,
    Y_training,
    X_testing_raw,
    Y_testing,
):
    training_delta_e = np.linalg.norm(
        X_training_raw[:, :3]
        - X_training_raw[:, 3:],
        axis=1,
    )

    testing_delta_e = np.linalg.norm(
        X_testing_raw[:, :3]
        - X_testing_raw[:, 3:],
        axis=1,
    )

    fitted_parameters = []

    for category in range(3):
        try:
            parameters, _ = curve_fit(
                nonlinear_regression_function,
                training_delta_e,
                Y_training[:, category],
                maxfev=(
                    REGRESSION_MAX_FUNCTION_EVALUATIONS
                ),
            )

        except Exception:
            parameters, _ = curve_fit(
                nonlinear_regression_function,
                training_delta_e,
                Y_training[:, category],
                p0=[
                    max(
                        float(
                            Y_training[:, category].mean()
                        ),
                        1e-3,
                    ),
                    -1.0,
                ],
                maxfev=(
                    REGRESSION_MAX_FUNCTION_EVALUATIONS
                    * 2
                ),
            )

        fitted_parameters.append(parameters)

    testing_raw_predictions = np.stack(
        [
            nonlinear_regression_function(
                testing_delta_e,
                *fitted_parameters[category],
            )
            for category in range(3)
        ],
        axis=1,
    )

    testing_predictions = (
        make_probability_distribution(
            testing_raw_predictions
        )
    )

    return (
        cross_entropy_numpy(
            Y_testing,
            testing_predictions,
        ),
        testing_predictions,
        fitted_parameters,
    )

###5-3.M1:Siamese

In [21]:
class M1Siamese(nn.Module):

    def __init__(self):
        super().__init__()
        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=1
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )
        return self.regressor(distance)

###5-4.M2:Siamese+Lab

In [22]:
class M2SiameseLab(nn.Module):

    def __init__(self):
        super().__init__()
        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=7
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )

        weighted_lab = (
            x * M2_LAB_WEIGHT
        )

        return self.regressor(
            torch.cat(
                [
                    distance,
                    weighted_lab,
                ],
                dim=1,
            )
        )

###5-5.M3:Siamese+ΔE

In [23]:
class M3SiameseDeltaE(nn.Module):

    def __init__(
        self,
        delta_e_min,
        delta_e_max,
    ):
        super().__init__()
        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=2
        )
        self.delta_e_min = float(
            delta_e_min
        )
        self.delta_e_max = float(
            delta_e_max
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )

        delta_e = torch.linalg.norm(
            x[:, :3] - x[:, 3:],
            dim=1,
            keepdim=True,
        )

        delta_e = (
            delta_e - self.delta_e_min
        ) / (
            self.delta_e_max
            - self.delta_e_min
            + 1e-8
        )

        return self.regressor(
            torch.cat(
                [
                    distance,
                    delta_e,
                ],
                dim=1,
            )
        )

###5-6.M4:Siamese+ΔE+Lab

In [24]:
class M4SiameseDeltaELab(nn.Module):

    def __init__(
        self,
        delta_e_min,
        delta_e_max,
    ):
        super().__init__()
        self.encoder = SharedEncoder()
        self.regressor = build_regressor(
            input_size=8
        )
        self.delta_e_min = float(
            delta_e_min
        )
        self.delta_e_max = float(
            delta_e_max
        )

    def forward(self, x):
        distance = calculate_siamese_distance(
            self.encoder,
            x,
        )

        delta_e = torch.linalg.norm(
            x[:, :3] - x[:, 3:],
            dim=1,
            keepdim=True,
        )

        delta_e = (
            delta_e - self.delta_e_min
        ) / (
            self.delta_e_max
            - self.delta_e_min
            + 1e-8
        )

        weighted_lab = (
            x * M4_LAB_WEIGHT
        )

        return self.regressor(
            torch.cat(
                [
                    distance,
                    delta_e,
                    weighted_lab,
                ],
                dim=1,
            )
        )

##6.Training and evaluation

In [25]:
def evaluate_neural_model(
    model,
    loader,
    return_predictions=False,
):
    model.eval()

    total_loss = 0.0
    total_samples = 0
    probability_batches = []

    with torch.no_grad():
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(
                DEVICE
            )
            Y_batch = Y_batch.to(
                DEVICE
            )

            logits = model(X_batch)
            loss = criterion(
                logits,
                Y_batch,
            )

            batch_size = X_batch.size(0)

            total_loss += (
                loss.item() * batch_size
            )
            total_samples += batch_size

            if return_predictions:
                probability_batches.append(
                    torch.softmax(
                        logits,
                        dim=1,
                    ).cpu().numpy()
                )

    average_loss = (
        total_loss / total_samples
    )

    if return_predictions:
        return (
            average_loss,
            np.concatenate(
                probability_batches,
                axis=0,
            ),
        )

    return average_loss


def train_neural_model(
    fold_number,
    model_name,
    model_factory,
    X_training,
    Y_training,
    X_testing,
    Y_testing,
):
    set_seed(SEED)

    model = model_factory().to(
        DEVICE
    )

    training_loader = make_loader(
        X_training,
        Y_training,
        shuffle=True,
        noise_std=INPUT_NOISE_STD,
        seed=SEED,
    )

    testing_loader = make_loader(
        X_testing,
        Y_testing,
        shuffle=False,
        noise_std=0.0,
        seed=SEED,
    )

    if WEIGHT_DECAY > 0:
        optimizer = optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
    else:
        optimizer = optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
        )

    if USE_SCHEDULER:
        scheduler = (
            optim.lr_scheduler
            .ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=SCHEDULER_FACTOR,
                patience=SCHEDULER_PATIENCE,
                threshold=SCHEDULER_THRESHOLD,
                min_lr=MIN_LEARNING_RATE,
            )
        )
    else:
        scheduler = None

    history_rows = []

    print()
    print(
        f"Fold {fold_number} | {model_name}"
    )

    for epoch in range(
        1,
        EPOCHS + 1,
    ):
        model.train()

        total_training_loss = 0.0
        total_samples = 0
        epoch_start = time.time()

        for X_batch, Y_batch in training_loader:
            X_batch = X_batch.to(
                DEVICE
            )
            Y_batch = Y_batch.to(
                DEVICE
            )

            optimizer.zero_grad()

            logits = model(X_batch)

            loss = criterion(
                logits,
                Y_batch,
            )

            loss.backward()
            optimizer.step()

            batch_size = X_batch.size(0)

            total_training_loss += (
                loss.item() * batch_size
            )
            total_samples += batch_size

        training_loss = (
            total_training_loss
            / total_samples
        )

        if scheduler is not None:
            scheduler.step(
                training_loss
            )

        history_rows.append({
            "Fold": fold_number,
            "Model": model_name,
            "Epoch": epoch,
            "Training loss":
                training_loss,
            "Learning rate":
                optimizer.param_groups[0]["lr"],
        })

        if (
            epoch == 1
            or epoch % PRINT_EVERY == 0
            or epoch == EPOCHS
        ):
            print(
                f"Epoch {epoch:4d} "
                f"train={training_loss:.6f} "
                f"lr={optimizer.param_groups[0]['lr']:.2e} "
                f"time={time.time() - epoch_start:.3f}s"
            )

    (
        test_loss,
        predictions,
    ) = evaluate_neural_model(
        model,
        testing_loader,
        return_predictions=True,
    )

    if SAVE_MODEL_WEIGHTS:
        torch.save(
            model.state_dict(),
            OUTPUT_DIR
            / (
                f"fold{fold_number}_"
                f"{model_name}_final_state.pt"
            ),
        )

    return (
        test_loss,
        predictions,
        pd.DataFrame(
            history_rows
        ),
        model,
    )


def release_model(model):
    del model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

##7.Execution

In [26]:
def bold_column_minimum(column):
    minimum = column.min()

    return [
        (
            "font-weight:bold"
            if np.isclose(
                value,
                minimum,
                rtol=1e-10,
                atol=1e-12,
            )
            else ""
        )
        for value in column
    ]


def run_experiment():
    result_records = []
    history_frames = []
    prediction_frames = []
    naive_records = []
    regression_records = []

    for fold_number, test_light in enumerate(
        UNIQUE_ILLUMINATIONS,
        start=1,
    ):
        test_mask = (
            ILLUMINATION_LABELS
            == test_light
        )

        train_indices = np.where(
            ~test_mask
        )[0]

        test_indices = np.where(
            test_mask
        )[0]

        X_training_raw = X_raw[
            train_indices
        ]
        Y_training = Y[
            train_indices
        ]
        X_testing_raw = X_raw[
            test_indices
        ]
        Y_testing = Y[
            test_indices
        ]

        if NORMALIZATION_SCOPE == "full_d1":
            normalization_reference = X_raw

        elif NORMALIZATION_SCOPE == "training_only":
            normalization_reference = (
                X_training_raw
            )

        else:
            raise ValueError(
                "NORMALIZATION_SCOPE must be "
                "'full_d1' or 'training_only'."
            )

        feature_min = normalization_reference.min(
            axis=0
        )
        feature_max = normalization_reference.max(
            axis=0
        )

        def normalize_features(X):
            return (
                X - feature_min
            ) / (
                feature_max
                - feature_min
                + 1e-8
            )

        X_training = normalize_features(
            X_training_raw
        ).astype(np.float32)

        X_testing = normalize_features(
            X_testing_raw
        ).astype(np.float32)

        normalized_reference = normalize_features(
            normalization_reference
        )

        delta_reference = np.linalg.norm(
            normalized_reference[:, :3]
            - normalized_reference[:, 3:],
            axis=1,
        )

        delta_e_min = float(
            delta_reference.min()
        )
        delta_e_max = float(
            delta_reference.max()
        )

        column_name = (
            f"Test on light {fold_number}"
        )

        print()
        print("#" * 80)
        print(
            column_name,
            "| actual label:",
            test_light,
        )

        #Naive
        (
            naive_loss,
            naive_predictions,
            naive_distribution,
        ) = run_naive(
            Y_training,
            Y_testing,
        )

        result_records.append({
            "Method": "Naive Prediction",
            "Condition": column_name,
            "Loss": naive_loss,
        })

        naive_records.append({
            "Condition": column_name,
            "Actual light label": str(
                test_light
            ),
            "Probability 0":
                float(naive_distribution[0]),
            "Probability 1":
                float(naive_distribution[1]),
            "Probability 2":
                float(naive_distribution[2]),
        })

        prediction_frames.append(
            pd.DataFrame({
                "Condition": column_name,
                "Sample index": test_indices,
                "Method": "Naive Prediction",
                "True 0": Y_testing[:, 0],
                "True 1": Y_testing[:, 1],
                "True 2": Y_testing[:, 2],
                "Predicted 0":
                    naive_predictions[:, 0],
                "Predicted 1":
                    naive_predictions[:, 1],
                "Predicted 2":
                    naive_predictions[:, 2],
            })
        )

        #Regression
        (
            regression_loss,
            regression_predictions,
            regression_parameters,
        ) = run_regression(
            X_training_raw,
            Y_training,
            X_testing_raw,
            Y_testing,
        )

        result_records.append({
            "Method": "ED Regression",
            "Condition": column_name,
            "Loss": regression_loss,
        })

        regression_records.append({
            "Condition": column_name,
            "Actual light label": str(
                test_light
            ),
            "Parameters": [
                parameters.tolist()
                for parameters
                in regression_parameters
            ],
        })

        prediction_frames.append(
            pd.DataFrame({
                "Condition": column_name,
                "Sample index": test_indices,
                "Method": "ED Regression",
                "True 0": Y_testing[:, 0],
                "True 1": Y_testing[:, 1],
                "True 2": Y_testing[:, 2],
                "Predicted 0":
                    regression_predictions[:, 0],
                "Predicted 1":
                    regression_predictions[:, 1],
                "Predicted 2":
                    regression_predictions[:, 2],
            })
        )

        neural_models = [
            (
                "Siamese",
                lambda: M1Siamese(),
            ),
            (
                "Siamese+Lab",
                lambda: M2SiameseLab(),
            ),
            (
                "Siamese+ΔE",
                lambda: M3SiameseDeltaE(
                    delta_e_min,
                    delta_e_max,
                ),
            ),
            (
                "Siamese+ΔE+Lab",
                lambda: M4SiameseDeltaELab(
                    delta_e_min,
                    delta_e_max,
                ),
            ),
        ]

        for method_name, model_factory in neural_models:
            (
                test_loss,
                predictions,
                history,
                model,
            ) = train_neural_model(
                fold_number=fold_number,
                model_name=method_name,
                model_factory=model_factory,
                X_training=X_training,
                Y_training=Y_training,
                X_testing=X_testing,
                Y_testing=Y_testing,
            )

            result_records.append({
                "Method": method_name,
                "Condition": column_name,
                "Loss": test_loss,
            })

            history[
                "Condition"
            ] = column_name
            history_frames.append(
                history
            )

            prediction_frames.append(
                pd.DataFrame({
                    "Condition": column_name,
                    "Sample index":
                        test_indices,
                    "Method": method_name,
                    "True 0":
                        Y_testing[:, 0],
                    "True 1":
                        Y_testing[:, 1],
                    "True 2":
                        Y_testing[:, 2],
                    "Predicted 0":
                        predictions[:, 0],
                    "Predicted 1":
                        predictions[:, 1],
                    "Predicted 2":
                        predictions[:, 2],
                })
            )

            release_model(model)

    result_long = pd.DataFrame(
        result_records
    )

    result_table = result_long.pivot(
        index="Method",
        columns="Condition",
        values="Loss",
    )

    result_table = result_table.reindex(
        METHOD_ORDER
    )

    result_table = result_table.reindex(
        columns=[
            "Test on light 1",
            "Test on light 2",
            "Test on light 3",
            "Test on light 4",
        ]
    )

    result_table.index.name = "Model"
    result_table.columns.name = None

    print()
    print("=" * 80)
    print("Final result table")

    styled_result = (
        result_table.style
        .format("{:.6f}")
        .apply(
            bold_column_minimum,
            axis=0,
        )
    )

    display(styled_result)

    histories = pd.concat(
        history_frames,
        ignore_index=True,
    )

    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    result_table.to_csv(
        OUTPUT_DIR
        / "experiment2_result_table.csv",
    )

    result_long.to_csv(
        OUTPUT_DIR
        / "experiment2_result_long.csv",
        index=False,
    )

    histories.to_csv(
        OUTPUT_DIR
        / "experiment2_training_history.csv",
        index=False,
    )

    if SAVE_PREDICTIONS:
        predictions.to_csv(
            OUTPUT_DIR
            / "experiment2_test_predictions.csv",
            index=False,
        )

    pd.DataFrame(
        naive_records
    ).to_csv(
        OUTPUT_DIR
        / "experiment2_naive_by_light.csv",
        index=False,
    )

    settings = {
        "D1_PATH": D1_PATH,
        "SEED": SEED,
        "ILLUMINATION_COLUMN":
            ILLUMINATION_COLUMN,
        "ILLUMINATION_SOURCE":
            ILLUMINATION_SOURCE,
        "NORMALIZATION_SCOPE":
            NORMALIZATION_SCOPE,
        "ENCODER_LAYERS":
            ENCODER_LAYERS,
        "REGRESSOR_LAYERS":
            REGRESSOR_LAYERS,
        "EPOCHS": EPOCHS,
        "BATCH_SIZE": BATCH_SIZE,
        "LEARNING_RATE":
            LEARNING_RATE,
        "INPUT_NOISE_STD":
            INPUT_NOISE_STD,
        "WEIGHT_DECAY":
            WEIGHT_DECAY,
        "M2_LAB_WEIGHT":
            M2_LAB_WEIGHT,
        "M4_LAB_WEIGHT":
            M4_LAB_WEIGHT,
        "USE_SCHEDULER":
            USE_SCHEDULER,
        "naive_by_light":
            naive_records,
        "regression_by_light":
            regression_records,
    }

    with open(
        OUTPUT_DIR
        / "experiment2_settings.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            settings,
            file,
            indent=2,
            ensure_ascii=False,
        )

    table_markdown = result_table.to_markdown(
        floatfmt=".6f"
    )

    summary_text = f'''
##9.Result summary

Each illumination condition was used once as the test subset, while the other three illumination conditions were used for training.

The methods were compared only within the same held-out illumination condition. Naive Prediction was recalculated separately in every condition from the corresponding training-subset mean distribution.

{table_markdown}

The complete result table, per-condition Naive distributions, regression parameters, training histories, predictions, model weights, and experiment settings were saved to the output directory.
'''

    display(
        Markdown(summary_text)
    )

    with open(
        OUTPUT_DIR
        / "experiment2_summary.md",
        "w",
        encoding="utf-8",
    ) as file:
        file.write(
            summary_text.strip()
        )

    print()
    print("Completed.")
    print("Files saved to:")
    print(OUTPUT_DIR)

    return {
        "result_table":
            result_table,
        "result_long":
            result_long,
        "histories":
            histories,
        "predictions":
            predictions,
        "settings":
            settings,
        "summary_text":
            summary_text,
    }


experiment2_outputs = (
    run_experiment()
)


################################################################################
Test on light 1 | actual label: Light 1

Fold 1 | Siamese
Epoch    1 train=0.951450 lr=5.00e-04 time=0.335s
Epoch   25 train=0.375786 lr=5.00e-04 time=0.321s
Epoch   50 train=0.347555 lr=5.00e-04 time=0.321s
Epoch   75 train=0.344126 lr=5.00e-04 time=0.320s
Epoch  100 train=0.343559 lr=5.00e-04 time=0.326s
Epoch  125 train=0.339711 lr=5.00e-04 time=0.318s
Epoch  150 train=0.339623 lr=5.00e-04 time=0.320s
Epoch  175 train=0.338394 lr=5.00e-04 time=0.322s
Epoch  200 train=0.337461 lr=5.00e-04 time=0.326s
Epoch  225 train=0.334921 lr=5.00e-04 time=0.345s
Epoch  250 train=0.339675 lr=5.00e-04 time=0.314s
Epoch  275 train=0.334583 lr=2.50e-04 time=0.313s
Epoch  300 train=0.335471 lr=2.50e-04 time=0.317s

Fold 1 | Siamese+Lab
Epoch    1 train=0.862367 lr=5.00e-04 time=0.331s
Epoch   25 train=0.356989 lr=5.00e-04 time=0.321s
Epoch   50 train=0.343031 lr=5.00e-04 time=0.327s
Epoch   75 train=0.345246 lr=5.00e-04 

,Test on light 1,Test on light 2,Test on light 3,Test on light 4
Model,,,,
Naive Prediction,0.824602,0.603480,0.484443,0.380684
ED Regression,0.652675,0.514800,0.396545,0.336582
Siamese,0.606684,0.461799,0.499631,0.399357
Siamese+Lab,0.589597,0.482718,0.411519,0.368658
Siamese+ΔE,0.609700,0.467915,0.425138,0.334270
Siamese+ΔE+Lab,0.586392,0.472693,0.357872,0.300314



##9.Result summary

Each illumination condition was used once as the test subset, while the other three illumination conditions were used for training.

The methods were compared only within the same held-out illumination condition. Naive Prediction was recalculated separately in every condition from the corresponding training-subset mean distribution.

| Model            |   Test on light 1 |   Test on light 2 |   Test on light 3 |   Test on light 4 |
|:-----------------|------------------:|------------------:|------------------:|------------------:|
| Naive Prediction |          0.824602 |          0.603480 |          0.484443 |          0.380684 |
| ED Regression    |          0.652675 |          0.514800 |          0.396545 |          0.336582 |
| Siamese          |          0.606684 |          0.461799 |          0.499631 |          0.399357 |
| Siamese+Lab      |          0.589597 |          0.482718 |          0.411519 |          0.368658 |
| Siamese+ΔE       |          0.609700 |          0.467915 |          0.425138 |          0.334270 |
| Siamese+ΔE+Lab   |          0.586392 |          0.472693 |          0.357872 |          0.300314 |

The complete result table, per-condition Naive distributions, regression parameters, training histories, predictions, model weights, and experiment settings were saved to the output directory.



Completed.
Files saved to:
/content/drive/MyDrive/CVIU_Experiment2_Leave_One_Illumination_Out
